# Scaled Excess Entropy Production vs. System Size — Arrhenius Pump Generator

This notebook replicates the scaling analysis from the bottom of `ctmc_tutorial.ipynb`,
using the **scaled** excess entropy production metric:

$$\frac{\sigma - \sigma_{\mathrm{MEPS}}}{\sigma_{\mathrm{MEPS}}}$$

for the Arrhenius pump generator across system sizes from $S=3$ to $S=729$.

The main result: as $S$ grows, the NESS excess approaches zero — large systems nearly minimize entropy production.

In [ ]:
import sys, os
sys.path.append(os.path.join(os.path.dirname(os.getcwd())))

from ctmc import ContinuousTimeMarkovChain as MC
from ctmc import arrhenius_pump_generator, HAS_JAX

import numpy as np
import time
import warnings
import matplotlib.pyplot as plt

%matplotlib inline

rc_dict = {
    'font.size': 14,
    'axes.labelsize': 'large',
    'legend.fontsize': 'small',
    'figure.autolayout': True,
    'mathtext.fontset': 'stix',
    'font.family': 'STIXGeneral',
    'figure.dpi': 120,
}
for k, v in rc_dict.items():
    plt.rcParams[k] = v

# suppress numpy divide-by-zero warnings from MEPS convergence internals
warnings.filterwarnings('ignore', category=RuntimeWarning, message='.*divide by zero.*')
warnings.filterwarnings('ignore', category=RuntimeWarning, message='.*invalid value.*')

print(f"JAX available: {HAS_JAX}")

## 1. Sweep over system sizes

We use the Arrhenius pump generator with 80% of transitions pumped at strength 4
(matching the tutorial parameters). For each system size $S$, we generate many
independent systems and compute NESS, MEPS, uniform, and random-state EPR values.

In [ ]:
%%time

sizes = [3**(i+1) for i in range(6)]   # [3, 9, 27, 81, 243, 729]
trials = [300, 300, 300, 200, 50, 10]  # fewer trials at large S

# pump parameters (matching tutorial)
pump_percent = 0.8
pump_strength = 4

pump_output = {}

for s, trial in zip(sizes, trials):
    n_pumps = max(1, int(pump_percent * (s**2 - s) / 2))
    print(f'S={s:>4d}, N={trial:>3d}, n_pumps={n_pumps} ...', end=' ', flush=True)

    machine = MC(S=s, N=trial, generator=arrhenius_pump_generator,
                 pump_strength=pump_strength, n_pumps=n_pumps)

    ness = machine.get_ness()
    meps = machine.get_meps()
    unif = machine.get_uniform()
    rand = machine.get_random_state()

    pump_output[s] = {
        'ness': machine.get_epr(ness),
        'meps': machine.get_epr(meps),
        'unif': machine.get_epr(unif),
        'rand': machine.get_epr(rand),
    }
    excess = pump_output[s]['ness'] / pump_output[s]['meps'] - 1
    print(f'done.  median scaled excess = {np.nanmedian(excess):.4f}')

print('\nAll sweeps complete.')

## 2. Scatter plots: scaled excess vs. MEPS EPR at each system size

Each panel shows one system size. Points are individual random systems.
As $S$ increases, the NESS cloud drops toward zero — the steady state
increasingly resembles the minimum entropy production state.

In [ ]:
state_types = ['ness', 'rand', 'unif']
state_colors = {'ness': 'tab:blue', 'rand': 'tab:orange', 'unif': 'tab:green'}

fig, axs = plt.subplots(1, len(sizes), figsize=(3.5 * len(sizes), 5),
                         sharey=True, sharex=True)

for ax, s in zip(axs, sizes):
    data = pump_output[s]
    meps_epr = data['meps']

    for st in state_types:
        scaled_excess = (data[st] - meps_epr) / meps_epr
        # filter to positive values for log scale
        valid = scaled_excess > 0
        ax.scatter(meps_epr[valid], scaled_excess[valid],
                   c=state_colors[st], label=st, alpha=0.15, s=12,
                   edgecolors='none')

    ax.set_title(f'S = {s}')
    ax.set_yscale('log')
    ax.set_xscale('log')
    ax.set_xlabel('$\\sigma_{\\mathrm{MEPS}}$')

axs[0].set_ylabel('$(\\sigma - \\sigma_{\\mathrm{MEPS}})\\;/\\;\\sigma_{\\mathrm{MEPS}}$')
axs[-1].legend(markerscale=3, framealpha=0.9)
fig.suptitle('Scaled excess EPR — Arrhenius pump generator\n'
             f'(pump fraction = {pump_percent}, strength = {pump_strength})',
             fontsize=14, y=1.06)
plt.tight_layout()

## 3. Summary: mean scaled excess vs. system size

This collapses the scatter into a single curve per state type, showing how the
excess entropy production scales with system size. Error bars show $\pm 1\sigma$.

The key prediction of the paper (Theorem 1): NESS excess decays as $O(1/S)$.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for st in state_types:
    means = []
    stds = []
    for s in sizes:
        data = pump_output[s]
        excess = (data[st] - data['meps']) / data['meps']
        # clean: remove non-finite or negative values
        excess = excess[np.isfinite(excess) & (excess > 0)]
        means.append(np.mean(excess))
        stds.append(np.std(excess))

    means = np.array(means)
    stds = np.array(stds)
    ax.errorbar(sizes, means, yerr=stds, marker='D', markersize=5,
                capsize=3, label=st, c=state_colors[st], linewidth=1.5)

# reference line: 1/S scaling
s_arr = np.array(sizes, dtype=float)
ref = means[0] * sizes[0] / s_arr  # anchored to first NESS point
ax.plot(s_arr, ref, 'k--', alpha=0.4, label='$\\propto 1/S$')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of states $S$')
ax.set_ylabel('$(\\sigma - \\sigma_{\\mathrm{MEPS}})\\;/\\;\\sigma_{\\mathrm{MEPS}}$')
ax.set_title('Scaling of excess EPR — Arrhenius pump generator')
ax.legend()
plt.tight_layout()

## 4. Effect of pump strength

How does the pump strength affect the convergence? We repeat the sweep at
several pump strengths and overlay the NESS excess curves.

In [ ]:
%%time

strengths = [1, 2, 4, 8]
strength_results = {}

for ps in strengths:
    print(f'\n=== pump_strength = {ps} ===')
    strength_results[ps] = {}

    for s, trial in zip(sizes, trials):
        n_pumps = max(1, int(pump_percent * (s**2 - s) / 2))
        print(f'  S={s:>4d}, N={trial:>3d} ...', end=' ', flush=True)

        machine = MC(S=s, N=trial, generator=arrhenius_pump_generator,
                     pump_strength=ps, n_pumps=n_pumps)

        ness_epr = machine.get_epr(machine.get_ness())
        meps_epr = machine.get_epr(machine.get_meps())

        excess = ness_epr / meps_epr - 1
        excess = excess[np.isfinite(excess) & (excess > 0)]

        strength_results[ps][s] = {
            'mean': np.mean(excess),
            'std': np.std(excess),
            'n': len(excess),
        }
        print(f'done. mean excess = {np.mean(excess):.4f} ({len(excess)} valid)')

print('\nAll strength sweeps complete.')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

for i, ps in enumerate(strengths):
    means = [strength_results[ps][s]['mean'] for s in sizes]
    stds = [strength_results[ps][s]['std'] for s in sizes]
    ax.errorbar(sizes, means, yerr=stds, marker='D', markersize=5,
                capsize=3, label=f'strength = {ps}', c=colors[i], linewidth=1.5)

# reference line
s_arr = np.array(sizes, dtype=float)
ref_anchor = strength_results[strengths[-1]][sizes[0]]['mean']
ref = ref_anchor * sizes[0] / s_arr

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of states $S$')
ax.set_ylabel('$(\\sigma_{\\mathrm{NESS}} - \\sigma_{\\mathrm{MEPS}})\\;/\\;\\sigma_{\\mathrm{MEPS}}$')
ax.set_title(f'NESS excess vs. system size at different pump strengths\n'
             f'(pump fraction = {pump_percent})')
ax.legend()
plt.tight_layout()

## Key Takeaways

- The scaled excess entropy production $\sigma_{\text{NESS}}/\sigma_{\text{MEPS}} - 1$ decreases systematically with system size $S$
- This holds across different pump strengths — the convergence is a robust feature of large interconnected systems
- The uniform and random states also converge, but more slowly than the NESS
- The $1/S$ reference line provides a guide for the decay rate (Theorem 1 from the paper)

## 5. Method Comparison: Euler vs JAX L-BFGS

We rerun the sweep using both MEPS methods on the same systems and compare
solution quality (EPR achieved) and wall-clock time. Both methods receive
the NESS as initial guess for a fair comparison. This is just a check to make sure that both methods would have agreed. At this point, jax is the way to go for any moderately large batch or state size. I may even stop supporting euler soon

In [ ]:
%%time

comparison_sizes = [3, 9, 27, 81, 243, 729]
comparison_trials = [100, 100, 50, 30, 20, 12]

comparison_output = {}

for s, trial in zip(comparison_sizes, comparison_trials):
    n_pumps = max(1, int(pump_percent * (s**2 - s) / 2))
    print(f'S={s:>4d}, N={trial:>3d}, n_pumps={n_pumps} ...', flush=True)

    machine = MC(S=s, N=trial, generator=arrhenius_pump_generator,
                 pump_strength=pump_strength, n_pumps=n_pumps)
    ness = machine.get_ness()

    # Euler method — pass state=ness for fair comparison
    t0 = time.perf_counter()
    meps_euler = machine.get_meps(method='euler', state=ness)
    t_euler = time.perf_counter() - t0
    epr_euler = machine.get_epr(meps_euler)

    # JAX method — same initial state
    t0 = time.perf_counter()
    meps_jax = machine.get_meps(method='jax', state=ness)
    t_jax = time.perf_counter() - t0
    epr_jax = machine.get_epr(meps_jax)

    epr_ness = machine.get_epr(ness)

    comparison_output[s] = {
        'epr_euler': epr_euler,
        'epr_jax': epr_jax,
        'epr_ness': epr_ness,
        't_euler': t_euler,
        't_jax': t_jax,
    }

    jax_better = np.sum(epr_jax < epr_euler - 1e-12)
    euler_better = np.sum(epr_euler < epr_jax - 1e-12)
    print(f'  Euler: {t_euler:.2f}s | JAX: {t_jax:.2f}s | '
          f'JAX lower in {jax_better}/{trial}, Euler lower in {euler_better}/{trial}')

print('\nComparison sweep complete.')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

s_arr = np.array(comparison_sizes)
t_euler = [comparison_output[s]['t_euler'] for s in comparison_sizes]
t_jax = [comparison_output[s]['t_jax'] for s in comparison_sizes]

x = np.arange(len(s_arr))
width = 0.35
ax1.bar(x - width/2, t_euler, width, label='Euler', color='tab:blue')
ax1.bar(x + width/2, t_jax, width, label='JAX L-BFGS', color='tab:red')
ax1.set_xticks(x)
ax1.set_xticklabels(s_arr)
ax1.set_xlabel('System size $S$')
ax1.set_ylabel('Wall-clock time (s)')
ax1.set_title('Computation time')
ax1.legend()
ax1.set_yscale('log')

# Speedup factor
speedup = np.array(t_euler) / np.array(t_jax)
ax2.plot(s_arr, speedup, 'D-', color='tab:purple', markersize=6)
ax2.axhline(1, color='k', ls='--', lw=0.8)
ax2.set_xlabel('System size $S$')
ax2.set_ylabel('Speedup (Euler time / JAX time)')
ax2.set_title('Relative speedup')
ax2.set_xscale('log')

fig.suptitle('Timing comparison: Euler vs JAX L-BFGS', fontsize=13, y=1.02)
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for method, key, color, marker, ls in [
    ('Euler', 'epr_euler', 'tab:blue', 'o', '-'),
    ('JAX', 'epr_jax', 'tab:red', 's', '--'),
]:
    means, stds = [], []
    for s in comparison_sizes:
        d = comparison_output[s]
        excess = (d['epr_ness'] - d[key]) / d[key]
        excess = excess[np.isfinite(excess) & (excess > 0)]
        means.append(np.mean(excess))
        stds.append(np.std(excess))

    means = np.array(means)
    stds = np.array(stds)
    ax.errorbar(comparison_sizes, means, yerr=stds, marker=marker,
                capsize=3, label=f'MEPS ({method})', c=color,
                linewidth=1.5, ls=ls, markersize=5)

# 1/S reference
s_arr = np.array(comparison_sizes, dtype=float)
ref = means[0] * comparison_sizes[0] / s_arr
ax.plot(comparison_sizes, ref, 'k:', alpha=0.4, label='$\\propto 1/S$')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of states $S$')
ax.set_ylabel('$(\\sigma_{\\mathrm{NESS}} - \\sigma_{\\mathrm{MEPS}})\\;/\\;'
              '\\sigma_{\\mathrm{MEPS}}$')
ax.set_title('Scaled NESS excess — Euler vs JAX L-BFGS')
ax.legend()
plt.tight_layout()